In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import sqlite3
import re

# Connect to the database
conn = sqlite3.connect("./data/nutrition.db")
cur = conn.cursor()

In [2]:
# Set up the db
wal_table = pd.read_csv("./data/wal_nutrient.csv")
wf_table = pd.read_csv("./data/wf_nutrient.csv")

wal_table.to_sql("wal_nutrient", conn, if_exists="replace", index=False)
wf_table.to_sql("wf_nutrient", conn, if_exists="replace", index=False)

1681

In [2]:
def normalize_text(str):
    if pd.isna(str):
        return ""
    str = str.lower()
    str = re.sub(r'[^a-z0-9\s]', ' ', str)
    str = re.sub(r'\s+', ' ', str).strip()
    return str

conn.create_function('normalize_text', 1, normalize_text)

In [9]:
search_name = widgets.Text(
    placeholder='Enter a product name',
    description='Name:',
    continuous_update=False,
)
search_brand = widgets.Text(
    placeholder='Enter a brand name',
    description='Brand:',
    continuous_update=False,
)
search_nutrient = widgets.Text(
    placeholder='Enter a nutrient name',
    description='Nutrient:',
    continuous_update=False,
)
out = widgets.Output()

def update(_change):
    wal_df = pd.read_sql_query("""
                       SELECT clean_desc, name, MIN(amount) as amount, MIN(price_retail) as price
                       FROM wal_nutrient
                       WHERE product_name LIKE ? AND clean_brand LIKE ? AND normalize_text(name) LIKE ?
                       GROUP BY clean_desc, name
                       """, conn, params=(f"%{search_name.value}%", f"%{normalize_text(search_brand.value)}%", f"%{normalize_text(search_nutrient.value)}%"))
    
    wf_df = pd.read_sql_query("""
                       SELECT clean_desc, name, MIN(amount) as amount, MIN(price) as price
                       FROM wf_nutrient
                       WHERE product_name LIKE ? AND clean_brand LIKE ? AND normalize_text(name) LIKE ?
                       GROUP BY clean_desc, name
                       """, conn, params=(f"%{search_name.value}%", f"%{normalize_text(search_brand.value)}%", f"%{normalize_text(search_nutrient.value)}%"))

    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))

        axes[0].scatter(wal_df.price, wal_df.amount)
        axes[0].set_xlabel('Price')
        axes[0].set_ylabel('Amount of Nutrient')
        axes[0].set_title('Walmart Price Distribution')

        axes[1].scatter(wf_df.price, wf_df.amount)
        axes[1].set_xlabel('Price')
        axes[1].set_ylabel('Amount of Nutrient')
        axes[1].set_title('Wholefoods Price Distribution')

        plt.tight_layout()
        plt.show()

search_name.observe(update, names='value')
search_brand.observe(update, names='value')
search_nutrient.observe(update, names='value')

display(widgets.VBox([search_name, search_brand, search_nutrient, out]))
update(None)